# In-Silico EEG Seizure Detection: Time-Series Feature Extraction & Wavelet Analysis
### Biological & Computational Overview

Epileptic seizures manifest as hypersynchronous neuronal discharges, visible in EEG as high-amplitude rhythmic spikes. Our approach decomposes EEG signals into physiologically meaningful frequency bands:
* **Delta (δ):** 0.5-4 Hz (Deep sleep / pathology)
* **Theta (θ):** 4-8 Hz (Drowsiness / hippocampal rhythms)
* **Alpha (α):** 8-13 Hz (Relaxed wakefulness)
* **Beta (β):** 13-30 Hz (Active cognition / hyperexcitability)
* **Gamma (γ):** 30-45 Hz (High-frequency discharges / ictal spikes)

## Step 1: Environment Setup & Installations
First, we install and import the required libraries. This notebook runs completely locally, but includes pip setup for convenience.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
import shap
from scipy import stats, signal as scipy_signal
from scipy.stats import entropy as shannon_entropy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set visualization aesthetics
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.dpi'] = 120
print(" All imports completed successfully!")

## Step 2: Data Acquisition & Preprocessing
We attempt to load the preprocessed EEG data from a Plotly mirror. If offline or unavailable, the notebook falls back to generating a realistic synthetic EEG dataset for demonstration.

In [ ]:
print("Loading dataset from GitHub mirror...")
url = "https://raw.githubusercontent.com/plotly/datasets/master/eeg_data.csv"

try:
    df = pd.read_csv(url)
    print(f" Dataset loaded from GitHub!")
    print(f"Shape: {df.shape}")
    
    # Adjust labels based on structure
    if 'class' in df.columns:
        y_binary = (df['class'] == 1).astype(int).values
        X = df.drop('class', axis=1)
    elif 'y' in df.columns:
        y_binary = (df['y'] == 1).astype(int).values
        X = df.drop('y', axis=1)
    else:
        X = df.iloc[:, :-1]
        y_raw = df.iloc[:, -1]
        y_binary = (y_raw == 1).astype(int).values
except Exception as e:
    print(f"Error: {e}")
    print(" Fallback: Creating a synthetic EEG-like dataset...")
    
    np.random.seed(42)
    n_samples = 500
    n_points = 178
    X_synthetic, y_synthetic = [], []
    
    for i in range(n_samples):
        if i < 100:  # Seizure class
            t = np.linspace(0, 1, n_points)
            # Seizure rhythmic pattern (8 Hz and 4 Hz) + noise
            signal_data = 5 * np.sin(2 * np.pi * 8 * t) + 3 * np.sin(2 * np.pi * 4 * t)
            signal_data += np.random.normal(0, 0.5, n_points)
            y_synthetic.append(1)
        else:  # Normal class
            t = np.linspace(0, 1, n_points)
            # Normal background + noise
            signal_data = 0.5 * np.sin(2 * np.pi * 10 * t) + np.random.normal(0, 1, n_points)
            y_synthetic.append(0)
        X_synthetic.append(signal_data)
        
    X = pd.DataFrame(X_synthetic)
    y_binary = np.array(y_synthetic)
    
    print(" Synthetic dataset created successfully!")

print(f"Features shape: {X.shape}")
print(f"Labels: {sum(y_binary)} seizure (ictal), {len(y_binary)-sum(y_binary)} non-seizure (normal)")

## Step 3: Frequency Band Decomposition (Butterworth Filtering)
We construct a bandpass Butterworth filter to extract the five primary frequency bands: Delta, Theta, Alpha, Beta, and Gamma.

In [ ]:
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = scipy_signal.butter(order, [low, high], btype='band')
    return b, a

def apply_bandpass_filter(data, lowcut, highcut, fs):
    b, a = butter_bandpass(lowcut, highcut, fs)
    return scipy_signal.filtfilt(b, a, data)

# Sampling Frequency (178 Hz)
FS = 178.0

bands = {
    'Delta': (0.5, 4.0),
    'Theta': (4.0, 8.0),
    'Alpha': (8.0, 13.0),
    'Beta': (13.0, 30.0),
    'Gamma': (30.0, 45.0)
}

print(" Filters defined for bands:", list(bands.keys()))

### Visualizing Signal Decomposition
Let's visualize the raw signal along with its filtered sub-bands for a seizure sample.

In [ ]:
sample_idx = 0  # Seizure sample
sample_eeg = X.iloc[sample_idx].values
time_axis = np.arange(len(sample_eeg)) / FS

fig, axes = plt.subplots(len(bands) + 1, 1, figsize=(14, 12), sharex=True)

# Plot Raw
axes[0].plot(time_axis, sample_eeg, color='black', lw=1.2)
axes[0].set_ylabel('Raw EEG (μV)')
axes[0].set_title(f"Signal Decomposition - Sample {sample_idx+1} ({'Seizure' if y_binary[sample_idx]==1 else 'Normal'})")
axes[0].grid(True, alpha=0.3)

# Plot bands
for idx, (band_name, (low, high)) in enumerate(bands.items(), start=1):
    filtered = apply_bandpass_filter(sample_eeg, low, high, FS)
    axes[idx].plot(time_axis, filtered, lw=1.0, color='C'+str(idx))
    axes[idx].set_ylabel(f'{band_name} (μV)\n{low}-{high} Hz')
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

### Power Spectral Density (PSD) Analysis
We compare Welch's PSD for a seizure vs. a non-seizure sample.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sz_idx = np.where(y_binary == 1)[0][0]
nsz_idx = np.where(y_binary == 0)[0][0]

for idx, label, color in [(sz_idx, 'Seizure (Ictal)', 'red'), (nsz_idx, 'Non-Seizure (Normal)', 'blue')]:
    freqs, psd = scipy_signal.welch(X.iloc[idx].values, FS, nperseg=64)
    ax.semilogy(freqs, psd, color=color, lw=2, label=label)

# Highlight bands
for band_name, (low, high) in bands.items():
    ax.axvspan(low, high, alpha=0.1, color='gray')
    ax.text((low+high)/2, ax.get_ylim()[1]*0.5, band_name, ha='center', fontsize=9)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (μV²/Hz)')
ax.set_title('Power Spectral Density Comparison')
ax.set_xlim([0, min(50, FS/2)])
ax.legend()
plt.tight_layout()
plt.show()

## Step 4: Feature Engineering
We extract time-domain statistics, wavelet-domain decomposition coefficients, and spectral features (relative band powers & spectral entropy) from the raw signal and each of the isolated bands.

In [ ]:
def extract_statistical_features(signal_data):
    std_val = np.std(signal_data)
    features = {
        'mean': np.mean(signal_data),
        'std': std_val,
        'variance': np.var(signal_data),
        'skewness': stats.skew(signal_data) if len(signal_data) > 2 and std_val > 1e-9 else 0.0,
        'kurtosis': stats.kurtosis(signal_data) if len(signal_data) > 3 and std_val > 1e-9 else 0.0,
        'rms': np.sqrt(np.mean(signal_data**2)),
        'peak_to_peak': np.ptp(signal_data),
        'shannon_entropy': shannon_entropy(np.abs(signal_data) + 1e-10)
    }
    return features

def extract_wavelet_features(signal_data, wavelet='db4', level=3):
    try:
        coeffs = pywt.wavedec(signal_data, wavelet, level=level)
        energies = [np.sum(np.array(c)**2) for c in coeffs]
        total_energy = sum(energies)
        ratios = [e/total_energy for e in energies] if total_energy > 0 else energies
        features = {f'wavelet_energy_L{i}': energies[i] for i in range(len(energies))}
        features['wavelet_total_energy'] = total_energy
        features['wavelet_entropy'] = shannon_entropy(ratios)
    except:
        features = {'wavelet_entropy': shannon_entropy(np.abs(signal_data) + 1e-10)}
    return features

def extract_spectral_features(signal_data, fs):
    freqs, psd = scipy_signal.welch(signal_data, fs, nperseg=min(128, len(signal_data)))
    psd_sum = np.sum(psd)
    if psd_sum > 0:
        psd_norm = psd / psd_sum
        spec_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-12)) / np.log2(len(psd_norm))
    else:
        spec_entropy = 0.0
    return {'spectral_entropy': spec_entropy, 'total_power': psd_sum}

def extract_all_features(eeg_signal, bands_dict, fs):
    all_features = {}
    # Raw features
    raw_stat = extract_statistical_features(eeg_signal)
    raw_spec = extract_spectral_features(eeg_signal, fs)
    for k, v in raw_stat.items(): all_features[f'raw_{k}'] = v
    for k, v in raw_spec.items(): all_features[f'raw_{k}'] = v
    
    # Relative Band Powers of raw signal
    freqs, psd = scipy_signal.welch(eeg_signal, fs, nperseg=min(128, len(eeg_signal)))
    band_powers = {}
    for b_name, (low, high) in bands_dict.items():
        idx = np.logical_and(freqs >= low, freqs <= high)
        band_powers[b_name] = np.sum(psd[idx]) if np.any(idx) else 0.0
    total_band_power = sum(band_powers.values())
    for b_name, p in band_powers.items():
        all_features[f'{b_name}_abs_power'] = p
        all_features[f'{b_name}_rel_power'] = p / total_band_power if total_band_power > 0 else 0.0
        
    # Filtered band features
    for b_name, (low, high) in bands_dict.items():
        if high < fs/2:
            try:
                filtered = apply_bandpass_filter(eeg_signal, low, high, fs)
                stat_feats = extract_statistical_features(filtered)
                for k, v in stat_feats.items(): all_features[f'{b_name}_{k}'] = v
                wave_feats = extract_wavelet_features(filtered)
                for k, v in wave_feats.items(): all_features[f'{b_name}_{k}'] = v
                spec_feats = extract_spectral_features(filtered, fs)
                for k, v in spec_feats.items(): all_features[f'{b_name}_{k}'] = v
            except:
                continue
    return all_features

### Extracting Features from the entire dataset

In [ ]:
print("Extracting features from all samples...")
feature_list = []
for i in range(len(X)):
    feature_list.append(extract_all_features(X.iloc[i].values, bands, FS))

feature_df = pd.DataFrame(feature_list)
feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(feature_df.mean())
feature_df['target'] = y_binary

print(f"Feature matrix shape: {feature_df.shape}")
print(f"Total features extracted per signal: {feature_df.shape[1] - 1}")

## Step 5: Model Training & Evaluation

In [ ]:
X_feat = feature_df.drop('target', axis=1)
y_feat = feature_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_feat, y_feat, test_size=0.2, stratify=y_feat, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

gb_model = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=4, 
    min_samples_split=10, subsample=0.8, random_state=42
)
gb_model.fit(X_train_scaled, y_train)

y_pred = gb_model.predict(X_test_scaled)
y_pred_proba = gb_model.predict_proba(X_test_scaled)[:, 1]

print("CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred, target_names=['Non-Seizure', 'Seizure']))

### Evaluation Curves (ROC & Precision-Recall)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
ax1.plot(fpr, tpr, 'b-', lw=2.5, label=f'Gradient Boosting (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1.2)
ax1.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend(loc='lower right')

# 2. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)
ax2.plot(recall, precision, 'g-', lw=2.5, label=f'PR Curve (AUC = {pr_auc:.3f})')
ax2.axhline(y=sum(y_test)/len(y_test), color='gray', linestyle='--', label='Baseline')
ax2.fill_between(recall, precision, alpha=0.1, color='green')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

## Step 6: Model Explainability (Gini & SHAP)

In [ ]:
# Gini Feature Importance
importances = gb_model.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
plt.barh(range(15), importances[indices], color='teal', edgecolor='black')
plt.yticks(range(15), [X_feat.columns[i] for i in indices])
plt.xlabel('Gini Impurity Reduction')
plt.title('Top 15 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Analysis
explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_test_scaled[:60])

if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Class 1 (seizure)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_scaled[:60], feature_names=X_feat.columns, show=False)
plt.title('SHAP Global Summary Plot', fontsize=14)
plt.tight_layout()
plt.show()

## Step 7: Wavelet Scalogram Visualization (CWT)
Continuous Wavelet Transform (CWT) maps the signal to the 2D time-frequency space, helping visual inspection of active epileptic discharge bands.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

seizure_sig = X.iloc[sz_idx].values
normal_sig = X.iloc[nsz_idx].values
scales = np.arange(1, 35)

for ax, sig, title in zip(axes, [seizure_sig, normal_sig], ['Seizure EEG (CWT)', 'Non-Seizure EEG (CWT)']):
    coefs, freqs = pywt.cwt(sig, scales, 'morl', sampling_period=1/FS)
    im = ax.imshow(np.abs(coefs), aspect='auto', origin='lower',
                   extent=[0, len(sig)/FS, 0, freqs[-1]], cmap='jet')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Magnitude')

plt.suptitle('Time-Frequency Wavelet Scalogram Comparison', fontsize=15)
plt.tight_layout()
plt.show()

## Step 8: Save Model Bundle & Local Serialization
We package and save the trained classifier, normalization scaler, feature names, test samples, and evaluation results. This allows the Streamlit dashboard to quickly load the entire configuration without retraining.

In [ ]:
import joblib

# Create model bundle dict
model_bundle = {
    'model': gb_model,
    'scaler': scaler,
    'feature_names': X_feat.columns.tolist(),
    'bands': bands,
    'fs': FS,
    'metrics': {
        'accuracy': float(gb_model.score(X_test_scaled, y_test)),
        'roc_auc': float(roc_auc_score(y_test, y_pred_proba)),
        'pr_auc': float(auc(recall, precision)),
        'sensitivity': float(confusion_matrix(y_test, y_pred)[1,1]/sum(y_test)),
        'specificity': float(confusion_matrix(y_test, y_pred)[0,0]/(len(y_test)-sum(y_test))),
        'f1_score': float(2*confusion_matrix(y_test, y_pred)[1,1]/(2*confusion_matrix(y_test, y_pred)[1,1]+confusion_matrix(y_test, y_pred)[0,1]+confusion_matrix(y_test, y_pred)[1,0])),
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
    },
    'test_data': {
        'raw_signals': X.loc[X_test.index].values.tolist(),
        'features': X_test.values.tolist(),
        'features_df': X_test,
        'scaled_features': X_test_scaled.tolist(),
        'labels': y_test.values.tolist()
    }
}

# Save locally
joblib.dump(model_bundle, 'eeg_model.pkl')
print(" Model bundle successfully saved to: eeg_model.pkl!")